# Deep Learning Math Lesson (7/17/26)

---

*Codecademy — Deep Learning with TensorFlow Path: Deep Learning Math. Concise grad-student notes.*

## Introduction

Before writing deep learning models, unbox the mechanics: this lesson covers the foundational math running through a neural network's inner workings — no deep linear algebra background required, just following the data's journey through the network.

## Scalars, Vectors, and Matrices

How data is represented, in increasing dimensionality:

| Object | Description | Example |
|---|---|---|
| **Scalar** | a single number | `x = 5` |
| **Vector** | a 1D array of numbers, indexed by position | `x = np.array([1, 2, 3])` |
| **Matrix** | a 2D grid of numbers (rows & columns), indexed by `[row, col]` | `x = np.array([[1,2,3],[4,5,6],[7,8,9]])` |

Scalars often represent tunable model quantities; vectors and matrices are the standard containers for data samples and datasets in NumPy.

In [ ]:
import numpy as np

scalar = 5

vector = np.array([1, 2, 3])

matrix = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])

print("scalar:", scalar)
print("vector:", vector, "| vector[1] =", vector[1])
print("matrix:\n", matrix, "\n| matrix[1, 2] =", matrix[1, 2])
# -> scalar: 5
# -> vector: [1 2 3] | vector[1] = 2
# -> matrix[1, 2] = 6 (row 1, col 2)

## Tensors

A **tensor** generalizes scalars, vectors, and matrices into a single concept: a **multidimensional array**. It's the core data structure in deep learning, giving more flexibility in the shape and dimensionality of the data you work with.

- scalar = 0-D tensor
- vector = 1-D tensor
- matrix = 2-D tensor
- tensor = *n*-D array, for any $n$

## Matrix Algebra

Core operations used throughout deep learning training:

- **Matrix addition** — element-wise: same-position entries add together (matrices must be the same shape).
- **Scalar multiplication** — a single scalar multiplies every element of the matrix.
- **Matrix multiplication** — the most involved: each output entry is a dot product of a row from the first matrix and a column from the second. Requires inner dimensions to match: $(m \times n) \cdot (n \times p) = (m \times p)$.
- **Transpose** — flips rows and columns; row $i$ of the original becomes column $i$ of the result. A $(m \times n)$ matrix transposes to $(n \times m)$.

Training a deep learning model is, at its core, repeatedly applying these operations to tensors.

In [ ]:
A = np.array([[3, 1], [2, 5]])
B = np.array([[4, 0], [1, 2]])

print("A + B (addition):\n", A + B)
# -> [[7 1] [3 7]]

print("\n2 * A (scalar mult):\n", 2 * A)
# -> [[6 2] [4 10]]

print("\nA @ B (matrix mult):\n", A @ B)
# -> [[13  2] [13 10]]

M = np.array([[6, 4, 24], [1, 3, 9]])  # 2x3
print("\nM.T (transpose):\n", M.T)
# -> 3x2: first row (6,4,24) becomes first column

## Neural Networks Concept Overview

**Layers:**

| Layer | Role |
|---|---|
| **Input** | one node per input feature (e.g. size, shape, nutrition for a food dataset) |
| **Hidden** | between input and output; adds complexity/learning capacity. Zero or more of them. |
| **Output** | final layer, produces the result. Exactly **one** per network. |

Nodes between layers are connected by **weights** — the learnable parameters that set connection strength.

**Weighted sum** (per layer, going from one layer to the next):

$$\text{weighted\_sum} = (\text{inputs} \cdot \text{weights}^T) + \text{bias}$$

**Activation function** is then applied to the weighted sum:

$$\text{Activation}(\text{weighted\_sum})$$

- Everything before the activation function is a **linear** transformation. Stacking linear layers without activation functions collapses to one big linear function — no more powerful than plain linear regression, no matter how many hidden layers.
- The activation function introduces **nonlinearity**, which is what gives deep networks their expressive power. It decides what signal "fires" forward to the next layer.

**Common activation functions:**
- **ReLU** (most common for hidden layers): $f(x) = \max(0, x)$ — 0 for negative inputs, identity slope for positive inputs.
- **Sigmoid** (often output layer): $f(x) = \dfrac{1}{1 + e^{-x}}$ — S-shaped curve, squashes output to $(0, 1)$.
- **Softmax** — also common for output layers (multi-class probabilities); covered in more depth later in the course.

In [ ]:
inputs = np.array([1.0, 2.0, 3.0])
weights = np.array([0.2, 0.5, -0.1])
bias = 0.4

weighted_sum = np.dot(inputs, weights.T) + bias
print("weighted sum:", weighted_sum)
# -> 1*0.2 + 2*0.5 + 3*-0.1 + 0.4 = 1.3

def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

print("ReLU(weighted_sum):   ", relu(weighted_sum))
print("Sigmoid(weighted_sum):", round(sigmoid(weighted_sum), 4))
# -> ReLU passes positive values through unchanged
# -> Sigmoid squashes the value into (0, 1)

## Loss Functions

Once the network produces an output, we need to measure how wrong it is. A **loss function** compares the predicted value against the actual (training-data) value.

| Loss | Used for | Formula |
|---|---|---|
| **Mean squared error (MSE)** | regression | average of $(\text{predicted} - \text{actual})^2$ over all points |
| **Cross-entropy loss** | classification | covered in more depth later in the course |

MSE is the same squared-distance-from-line-of-best-fit idea familiar from linear regression.

In [ ]:
predicted = np.array([2.5, 0.0, 2.1, 7.8])
actual = np.array([3.0, -0.5, 2.0, 7.0])

mse = np.mean((predicted - actual) ** 2)
print("MSE:", round(mse, 4))
# -> average squared error between predicted and actual

## Backpropagation

- **Forward propagation** — feeding inputs through the hidden layers to the output (everything covered so far).
- **Backpropagation** — after computing the loss, work *backward* through the network computing **gradients**: the rate of change of the loss function with respect to each weight.
- **Gradient descent** — uses those gradients to update each weight, nudging it in the direction that decreases the loss.

Conceptually: backprop figures out how much each individual weight contributed to the error, and gradient descent adjusts the weights accordingly to reduce that error over training iterations. (Full derivation isn't needed for this course — conceptual understanding is enough.)

## Gradient Descent

Graphically: the loss function is a surface, and we want its **minimum** (highest accuracy). Starting from a random point, gradient descent takes steps toward the **negative gradient** — "downhill."

Weight update rule:

$$\text{param}_{new} = \text{param}_{old} - \text{learning\_rate} \cdot \nabla(\text{loss}(\text{param}_{old}))$$

The **learning rate** controls step size, and picking it well is a balancing act:
- **Too large** → overshoots the minimum, can diverge entirely instead of converging.
- **Too small** → painfully slow, and more likely to get stuck in a local minimum without ever reaching the true optimum.

In [ ]:
# toy loss: L(w) = (w - 3)^2, minimum at w = 3
def loss(w):
    return (w - 3) ** 2

def grad(w):
    return 2 * (w - 3)

w = 0.0            # random starting point
learning_rate = 0.1

for step in range(10):
    w = w - learning_rate * grad(w)

print("w after 10 steps:", round(w, 4))
print("loss at that w:  ", round(loss(w), 6))
# -> w converges toward 3.0, loss shrinks toward 0

## Stochastic Gradient Descent (SGD)

Plain ("batch") gradient descent recomputes gradients over the **entire dataset** every iteration — with large datasets this is computationally brutal. Example: 100,000 data points, 5 parameters, 1000 iterations (**epochs**) → $100{,}000 \times 5 \times 1000 = 500{,}000{,}000$ computations.

**SGD** fixes this: instead of using the whole dataset each iteration, it picks a single **random data point** to compute the gradient from. Far cheaper per step, and still converges to accurate results.

## More Variants of Gradient Descent

| Variant | Idea | Why use it |
|---|---|---|
| **Batch GD** | full dataset per iteration | accurate, but computationally expensive |
| **SGD** | one random data point per iteration | cheap, but noisy/erratic updates |
| **Mini-batch GD** | small fixed-size batch per iteration | trade-off: smoother than SGD (less noise/outlier-sensitive), cheaper than full batch GD |
| **Adam** | adaptive learning rate *per parameter* | commonly used default in deep learning — adapts step size automatically instead of using one fixed learning rate for everything |

This is an actively evolving area — new optimizer variants keep appearing as the field improves efficiency and accuracy.